# Task 8a: MolE's full original protocol, 5 trials (GPU)

Rows: MolE, MolE + GPT-5.2, GPT-5.2 only, RDKit descriptors only. Per trial k, 5 configurations are drawn (seed k, shared by every row) from MolE's search space (n_estimators 30-1000, max_depth 5-100, eta 0.05-1, subsample 0.3-1), each fitted 5 times; the configuration with the best mean validation PR-AUC is selected. Raw features, no early stopping, no class weighting, pooled PR-AUC = auc(recall, precision), as in MolE's code. The GPU uses `hist`; MolE used `exact` (MolE alone: 0.7802 exact vs 0.7655 hist on CPU).

Every fit is appended to `fits.csv` and finished fits are skipped, so adding a row costs only that row's time.

**To add `rdkit_only` to a finished run: attach this notebook's previous output** (Input panel -> Add Input -> Notebook Output). Cell 4 then shows 125 fits for each of the three rows already done, and only `rdkit_only` runs (about 4 minutes). Without that attachment every row is recomputed from scratch, which is correct but costs ~80 minutes.

The paired statistics compare each row with `mole_only` from the same `fits.csv`, so all rows must come from the same GPU run: CPU and GPU `hist` disagree by up to 0.02 on this data, which is larger than the differences being measured.

**Setup:** attach the `gpt_mole_bundle.zip` dataset (no new version needed for this run), Accelerator = GPU, Internet on, then **Save Version -> Save & Run All**.

In [ ]:
# 1. Confirm XGBoost really trains on the GPU. XGBoost silently falls back to CPU
#    when no GPU is visible, so check for that warning rather than trusting the setting.
import subprocess, sys, warnings
import numpy as np
import xgboost as xgb

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("python", sys.version.split()[0], "| xgboost", xgb.__version__)
assert int(xgb.__version__.split(".")[0]) >= 2, "device='cuda' needs xgboost >= 2"

X = np.random.rand(2000, 20)
y = (X[:, 0] > 0.5).astype(int)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    xgb.train({"tree_method": "hist", "device": "cuda", "objective": "binary:logistic"},
              xgb.DMatrix(X, label=y), 10)
fallback = [str(w.message) for w in caught if "GPU" in str(w.message)]
assert not fallback, f"XGBoost fell back to CPU - turn on a GPU accelerator.\n{fallback}"
print("GPU training OK")

In [ ]:
# 2. Dependencies. build_train.py imports rdkit (unused by these runs) and reads the
#    SF3 workbook with openpyxl; install either only if the image lacks it.
import importlib.util

for module, package in [("rdkit", "rdkit"), ("openpyxl", "openpyxl")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import pandas, sklearn
print("pandas", pandas.__version__, "| scikit-learn", sklearn.__version__)

In [ ]:
# 3. Copy the bundle to a writable folder (inputs are read-only) and work from there.
#    Also restores finished runs from a previous version's output, if one is attached.
import glob, gzip, importlib, os, shutil, sys

MARKER = "scripts/run_mole_protocol_search.py"   # present only in the Task 8 bundle
found = [os.path.dirname(os.path.dirname(h))
         for h in glob.glob("/kaggle/input/**/notebooks/build_train.py", recursive=True)]
assert found, "Bundle not found - attach the dataset made from gpt_mole_bundle.zip."
# Several dataset versions can be attached at once, so use one that has the new files
# rather than whichever the glob happens to return first.
usable = [b for b in found if os.path.exists(os.path.join(b, MARKER))]
if not usable:
    listing = "\n".join(
        f"  {b}  (run_table1.py: {os.path.getsize(os.path.join(b, 'scripts/run_table1.py')):,} bytes)"
        for b in found)
    raise AssertionError(
        "The attached dataset is an OLD version - it does not contain " + MARKER + ".\n"
        "Attached bundle(s):\n" + listing + "\n"
        "Fix: upload the new gpt_mole_bundle.zip to the dataset via New Version, then in this\n"
        "notebook's Input panel remove the dataset and add it again so the newest version is\n"
        "used (run_table1.py is then ~12,463 bytes and data/gpt_guided_flat/ exists).")
BUNDLE = usable[0]
WORK = "/kaggle/working/gpt_mole"
# dirs_exist_ok refreshes code and data in place while keeping any results/ already
# computed, so a newer dataset version takes effect without restarting the kernel.
shutil.copytree(BUNDLE, WORK, dirs_exist_ok=True)
os.chdir(WORK)
print("bundle:", BUNDLE, "-> working in", os.getcwd())
if os.path.exists("MANIFEST.txt"):
    print(open("MANIFEST.txt").read())

sys.path.insert(0, "scripts")
import run_table1
importlib.reload(run_table1)   # in case an older copy was imported earlier in this kernel

# Kaggle decompresses .gz files inside an uploaded dataset, so x.tsv.gz arrives as
# x.tsv. Re-compress them to the names the scripts expect.
needed = [run_table1.SF3, run_table1.SPLIT, run_table1.MOLE, run_table1.PRESTWICK,
          run_table1.CHEMDESC, *run_table1.FREEFORM.values()]
for path in needed:
    if path.endswith(".gz") and not os.path.exists(path) and os.path.exists(path[:-3]):
        with open(path[:-3], "rb") as src, gzip.open(path, "wb") as dst:
            shutil.copyfileobj(src, dst)
        print("re-compressed", path)
missing = [p for p in needed if not os.path.exists(p)]
assert not missing, f"Missing inputs: {missing}"
print(f"all {len(needed)} inputs present")

OUT_ROOT = "results/mole_protocol_trials"
previous = [d for d in glob.glob("/kaggle/input/**/mole_protocol_trials", recursive=True)
            if not d.startswith(BUNDLE)]
for d in previous:
    shutil.copytree(d, OUT_ROOT, dirs_exist_ok=True)
    print("restored finished runs from", d)

In [ ]:
# 4. What came back from a previous run, if its output was attached. Rows already
#    complete are skipped below, so only the missing ones cost GPU time.
import pandas as pd
fits_csv = f"{OUT_ROOT}/fits.csv"
if os.path.exists(fits_csv):
    print(pd.read_csv(fits_csv).groupby("row").size().rename("fits").to_string(), "\n(125 = complete)")
else:
    print("no previous fits.csv - this will run every row from scratch (~80 min on GPU)")

In [ ]:
# 5. Run all fits (appends to fits.csv, so finished rows are skipped).
ROWS = "mole_only,mole_plus_gpt-5.2,gpt_only_gpt-5.2,rdkit_only"
!python -u scripts/run_mole_protocol_search.py --rows {ROWS} --trials 0,1,2,3,4 --device cuda --output_dir {OUT_ROOT}
assert _exit_code == 0, "run failed - see the output above"

# The paired statistics need mole_only in the same fits.csv, on the same GPU.
fits = pd.read_csv(fits_csv)
for row in ROWS.split(","):
    assert (fits.row == row).sum() == 125, f"{row}: {(fits.row == row).sum()} fits, expected 125"
print("all rows complete")

In [ ]:
# 6. Table: mean +/- sd over trials and paired difference vs MolE; then the selected configuration per trial.
!python scripts/run_mole_protocol_search.py --aggregate_only --output_dir {OUT_ROOT}
pd.set_option("display.width", 220)
display(pd.read_csv(f"{OUT_ROOT}/table_mole_protocol.csv").round(4))
pd.read_csv(f"{OUT_ROOT}/selected_per_trial.csv").round(4)

In [ ]:
# 7. Package for download (Output tab -> task8a_results.zip).
archive = shutil.make_archive("/kaggle/working/task8a_results", "zip", OUT_ROOT)
print(archive, f"{os.path.getsize(archive) / 1024 ** 2:.1f} MB")